# 🎓 EduMIND — LoRA Fine-Tuning LLM for Vietnamese↔English Translation
> **Dataset**: [VietMix](https://huggingface.co/datasets/razent/vietmix) — Code-mixed Vietnamese-English sentences  
> **Framework**: [UnSloth](https://github.com/unslothai/unsloth) + PEFT LoRA  
> **Target**: Fine-tune a causal LLM to translate code-mixed Viet↔Eng while **preserving technical/domain terms**  
> **Hardware**: Kaggle 2× NVIDIA T4 (Multi-GPU)  

---
## 🗺️ Notebook Structure
1. **Environment Setup** — Install UnSloth, verify GPUs
2. **Dataset Loading** — Load & explore VietMix from HuggingFace
3. **Data Preprocessing** — Build instruction-tuning prompt template
4. **Model Loading** — Load base LLM with 4-bit quantization
5. **LoRA Config** — Apply parameter-efficient fine-tuning adapters
6. **Training** — Fine-tune with `SFTTrainer` (multi-GPU via DDP)
7. **Evaluation** — BLEU/chrF score on held-out set
8. **Save & Export** — Merge adapters and push to HuggingFace Hub

> **⚙️ API Key Fallback**: If you run this without a fine-tuned model, the system falls back to Google Gemini / OpenAI API.

## 1️⃣ Environment Setup

In [ ]:
# ── Detect hardware ──────────────────────────────────────────────────────────
import subprocess
import sys

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu_info.stdout)

# Count GPUs
import torch
NUM_GPUS = torch.cuda.device_count()
print(f"\n✅ Found {NUM_GPUS} GPU(s)")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
# ── Install UnSloth + dependencies ─────────────────────────────────────────
# UnSloth optimizes memory & speed for LoRA fine-tuning on consumer GPUs
!pip install --quiet \
    "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" \
    datasets \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    sacrebleu \
    evaluate \
    huggingface_hub

print("✅ All dependencies installed")

In [ ]:
# ── Global Configuration ─────────────────────────────────────────────────────
import os

# ── Model choice (change if you want a different base LLM) ──────────────────
# Recommended choices for 2×T4 (15 GB VRAM each):
#   - "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   (best quality, fits 2×T4)
#   - "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
#   - "unsloth/llama-3-8b-Instruct-bnb-4bit"
#   - "unsloth/gemma-2-9b-it-bnb-4bit"          (slightly slower)
BASE_MODEL_ID   = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

# ── Output paths ────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/kaggle/working/edumind-vietmix-llm"
HF_REPO_ID      = "YOUR_HF_USERNAME/edumind-vietmix-qwen2.5-7b-lora"  # ← change this

# ── Training hyperparameters ─────────────────────────────────────────────────
MAX_SEQ_LEN     = 512      # max input+output token length
BATCH_SIZE      = 4        # per-device batch size
GRAD_ACCUM      = 4        # effective batch = BATCH_SIZE * GRAD_ACCUM * NUM_GPUS
EPOCHS          = 3
LEARNING_RATE   = 2e-4
WARMUP_RATIO    = 0.05
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.01

# ── LoRA hyperparameters ─────────────────────────────────────────────────────
LORA_R          = 16       # rank
LORA_ALPHA      = 32       # scaling factor = alpha/r → 2×
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Evaluation/Split ─────────────────────────────────────────────────────────
EVAL_SPLIT      = 0.05     # 5% of data for validation
SEED            = 42

# ── HuggingFace token (set as Kaggle Secret named HF_TOKEN) ─────────────────
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — model will not be pushed to Hub")
else:
    print("✅ HF_TOKEN found")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n📋 Config summary:")
print(f"   Base model : {BASE_MODEL_ID}")
print(f"   LoRA rank  : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Epochs     : {EPOCHS}")
print(f"   Batch size : {BATCH_SIZE} × grad_accum={GRAD_ACCUM} × {NUM_GPUS} GPU(s)")

## 2️⃣ Dataset Loading & Exploration

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load VietMix dataset from HuggingFace
print("📥 Loading VietMix dataset from HuggingFace...")
raw_dataset = load_dataset("razent/vietmix", trust_remote_code=True)
print("\n📊 Dataset info:")
print(raw_dataset)

# Inspect available splits and column names
print("\n🔍 Available splits:", list(raw_dataset.keys()))
split_name = list(raw_dataset.keys())[0]
print(f"\n📑 Columns in '{split_name}':", raw_dataset[split_name].column_names)

# Preview a few examples
df_preview = raw_dataset[split_name].to_pandas().head(10)
print(f"\n📋 First 10 examples:")
pd.set_option('display.max_colwidth', 120)
print(df_preview)

In [ ]:
# ── Deeper exploration ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

df = raw_dataset[split_name].to_pandas()
print(f"📊 Total examples: {len(df):,}")
print(f"\n📑 Column dtypes:\n{df.dtypes}")
print(f"\n📋 Sample null counts:\n{df.isnull().sum()}")

# Detect the actual text columns (vi and en)
# VietMix format: typically 'vi' (Vietnamese) and 'en' (English) columns
# or 'sentence1' and 'sentence2' — adapt based on actual schema
print("\n🔍 First 3 rows (all columns):")
for i, row in df.head(3).iterrows():
    print(f"\n--- Row {i} ---")
    for col, val in row.items():
        print(f"  [{col}]: {str(val)[:200]}")

## 3️⃣ Data Preprocessing — Build Instruction-Tuning Prompts

In [ ]:
# ── Identify correct column names ────────────────────────────────────────────
# VietMix has columns: 'vi' and 'en' (Vietnamese and English sentences)
# Adjust these if the actual column names differ after exploring above
COLUMNS = raw_dataset[split_name].column_names

# Auto-detect vi/en columns
VI_COL = None
EN_COL = None
for c in COLUMNS:
    if c.lower() in ("vi", "vietnamese", "source", "sentence1"):
        VI_COL = c
    elif c.lower() in ("en", "english", "target", "sentence2"):
        EN_COL = c

if VI_COL is None or EN_COL is None:
    # Fallback: assume first two columns
    VI_COL, EN_COL = COLUMNS[0], COLUMNS[1]
    print(f"⚠️  Could not auto-detect columns — assuming: VI={VI_COL}, EN={EN_COL}")
else:
    print(f"✅ Detected columns: VI='{VI_COL}', EN='{EN_COL}'")

# Preview
print(f"\nSample VI: {df[VI_COL].iloc[0]}")
print(f"Sample EN: {df[EN_COL].iloc[0]}")

In [ ]:
# ── Define EduMIND Translation Prompt Template ────────────────────────────────
# Key insight: we instruct the model to PRESERVE technical/domain terms (English)
# This is the core EduMIND requirement for bilingual education

SYSTEM_PROMPT = """You are EduMIND, an expert bilingual academic translator specializing \
in Vietnamese-English educational content. Your task is to translate text while:
1. Producing natural, fluent output in the target language
2. PRESERVING technical/domain-specific English terms as-is (do not translate them)
3. Maintaining academic tone and precision
4. Keeping proper nouns, abbreviations, and specialized terminology unchanged

Examples of terms to preserve: machine learning, deep learning, neural network, \
transformer, embedding, fine-tuning, dataset, API, GPU, CPU, etc."""


def format_vi_to_en_prompt(vi_text: str, en_text: str = None) -> dict:
    """Format a Vietnamese→English translation example as instruction-tuning prompt."""
    user_msg = f"Translate the following Vietnamese text to English:\n\n{vi_text}"
    if en_text is not None:
        # Training format (with answer)
        return {
            "text": (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n{en_text}<|im_end|>"
            )
        }
    else:
        # Inference format (without answer)
        return {
            "text": (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
        }


def format_en_to_vi_prompt(en_text: str, vi_text: str = None) -> dict:
    """Format an English→Vietnamese translation example as instruction-tuning prompt."""
    user_msg = f"Translate the following English text to Vietnamese, preserving all technical terms in English:\n\n{en_text}"
    if vi_text is not None:
        return {
            "text": (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n{vi_text}<|im_end|>"
            )
        }
    else:
        return {
            "text": (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
        }


# Test prompt formatting
sample = format_vi_to_en_prompt(
    vi_text=df[VI_COL].iloc[0],
    en_text=df[EN_COL].iloc[0]
)
print("📋 Sample training prompt:")
print("─" * 60)
print(sample["text"])
print("─" * 60)

In [ ]:
from datasets import Dataset, DatasetDict
import random

random.seed(SEED)

# ── Build instruction-tuning dataset ─────────────────────────────────────────
# Strategy: create BOTH directions (vi→en AND en→vi) from same pairs
# This doubles the dataset size and trains bidirectional translation

def build_instruction_dataset(hf_dataset, vi_col: str, en_col: str) -> Dataset:
    """Convert translation pairs into instruction-tuning examples."""
    examples = []
    
    for row in hf_dataset:
        vi = str(row[vi_col]).strip()
        en = str(row[en_col]).strip()
        
        # Skip empty or very short pairs
        if len(vi) < 5 or len(en) < 5:
            continue
        
        # Direction 1: Vietnamese → English
        examples.append(format_vi_to_en_prompt(vi, en))
        
        # Direction 2: English → Vietnamese (with domain term preservation)
        examples.append(format_en_to_vi_prompt(en, vi))
    
    return Dataset.from_list(examples)


print("⚙️  Building instruction-tuning dataset...")
full_dataset = build_instruction_dataset(raw_dataset[split_name], VI_COL, EN_COL)
print(f"✅ Total instruction examples: {len(full_dataset):,}")
print(f"   (2× original = {len(raw_dataset[split_name]):,} pairs × 2 directions)")

# ── Train/Validation split ───────────────────────────────────────────────────
split = full_dataset.train_test_split(test_size=EVAL_SPLIT, seed=SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"\n📊 Split summary:")
print(f"   Train : {len(train_dataset):,}")
print(f"   Eval  : {len(eval_dataset):,}")

# Token length analysis
sample_lengths = [len(ex["text"].split()) for ex in train_dataset.select(range(min(1000, len(train_dataset))))]
print(f"\n📏 Token length stats (word-based approx):")
print(f"   Mean   : {np.mean(sample_lengths):.0f}")
print(f"   Median : {np.median(sample_lengths):.0f}")
print(f"   Max    : {np.max(sample_lengths)}")
print(f"   p95    : {np.percentile(sample_lengths, 95):.0f}")

## 4️⃣ Load Base Model with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel

print(f"📥 Loading base model: {BASE_MODEL_ID}")
print("   (4-bit quantization via bitsandbytes — reduces VRAM by ~75%)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto-detect (bfloat16 on Ampere+, float16 on T4)
    load_in_4bit=True,   # QLoRA: 4-bit base model
    token=HF_TOKEN if HF_TOKEN else None,
)

print(f"\n✅ Model loaded successfully!")
print(f"   Architecture  : {model.config.model_type}")
print(f"   Hidden size   : {model.config.hidden_size}")
print(f"   Num layers    : {model.config.num_hidden_layers}")
print(f"   Vocab size    : {model.config.vocab_size:,}")

# VRAM usage after loading
for i in range(NUM_GPUS):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved  = torch.cuda.memory_reserved(i) / 1e9
    print(f"\n   GPU {i} VRAM: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")

## 5️⃣ Apply LoRA Adapters

In [ ]:
# ── Apply LoRA with UnSloth ───────────────────────────────────────────────────
# LoRA adds small trainable rank-decomposition matrices to attention layers
# Only ~1-5% of parameters are trained → fast & memory-efficient

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGETS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",            # no bias tuning
    use_gradient_checkpointing="unsloth",  # UnSloth's memory-efficient checkpointing
    random_state=SEED,
    use_rslora=True,        # Rank-Stabilized LoRA (better stability)
    loftq_config=None,
)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Parameter summary:")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"   Frozen params    : {total_params - trainable_params:,}")

## 6️⃣ Training with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Training arguments ───────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Batch & gradient
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # Multi-GPU strategy
    # Note: UnSloth handles device placement; DDP is enabled automatically
    # when launched with torchrun or Kaggle's multi-GPU environment
    
    # Learning rate schedule
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    
    # Epochs & saving
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # Precision
    fp16=not is_bfloat16_supported(),  # float16 on T4 (no bfloat16 support)
    bf16=is_bfloat16_supported(),
    
    # Logging
    logging_steps=50,
    report_to="none",  # disable wandb/tensorboard for simplicity
    
    # Sequence packing (UnSloth feature — speeds up training ~30%)
    # packing=True when using SFTTrainer
    
    seed=SEED,
    optim="adamw_8bit",  # 8-bit AdamW — saves VRAM vs float32 optimizer states
    max_grad_norm=1.0,
    
    # Disable model card push (we'll do it manually)
    push_to_hub=False,
)

print("✅ TrainingArguments configured")
eff_batch = BATCH_SIZE * GRAD_ACCUM * NUM_GPUS
print(f"   Effective batch size: {eff_batch}")
steps_per_epoch = len(train_dataset) // eff_batch
print(f"   Steps per epoch    : {steps_per_epoch:,}")
print(f"   Total steps        : {steps_per_epoch * EPOCHS:,}")

In [ ]:
# ── Build SFTTrainer ─────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=True,          # pack multiple short examples into one sequence
    args=training_args,
)

# Memory check before training
print("📊 VRAM usage before training:")
for i in range(NUM_GPUS):
    used = torch.cuda.memory_allocated(i) / 1e9
    total_vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.2f} / {total_vram:.1f} GB ({100*used/total_vram:.1f}%)")

In [ ]:
# ── Start Training ───────────────────────────────────────────────────────────
import time

print("🚀 Starting LoRA fine-tuning...")
print(f"   This will take approximately {steps_per_epoch * EPOCHS // 60 + 1} minutes on 2×T4")
print("─" * 60)

start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

print("\n✅ Training complete!")
print(f"   Total time      : {elapsed/60:.1f} minutes")
print(f"   Train loss      : {trainer_stats.training_loss:.4f}")
print(f"   Steps completed : {trainer_stats.global_step}")

## 7️⃣ Evaluation — BLEU & chrF Scores

In [ ]:
import evaluate
from unsloth import FastLanguageModel

# Switch to inference mode (faster)
FastLanguageModel.for_inference(model)

# Load metrics
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")


def translate_batch(texts_vi: list[str], max_new_tokens: int = 256) -> list[str]:
    """Run batch inference for vi→en translation."""
    prompts = [format_vi_to_en_prompt(t)["text"] for t in texts_vi]
    
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # greedy for evaluation
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only new tokens (strip the prompt)
    results = []
    for i, output in enumerate(outputs):
        input_len = inputs.input_ids[i].shape[0]
        new_tokens = output[input_len:]
        decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        results.append(decoded)
    
    return results


# ── Run evaluation on a subset of eval set ───────────────────────────────────
N_EVAL = min(200, len(eval_dataset))
eval_subset = eval_dataset.select(range(N_EVAL))

print(f"📊 Evaluating on {N_EVAL} examples...")

# Extract vi/en pairs from eval prompts
# We need original pairs — use the raw dataset eval split
raw_eval = raw_dataset[split_name].train_test_split(test_size=EVAL_SPLIT, seed=SEED)["test"]
raw_eval = raw_eval.select(range(min(N_EVAL // 2, len(raw_eval))))  # only vi→en examples

vi_texts = [str(row[VI_COL]) for row in raw_eval]
en_refs  = [[str(row[EN_COL])] for row in raw_eval]  # list of lists for sacrebleu

# Batch inference
EVAL_BATCH = 8
predictions = []
for start in range(0, len(vi_texts), EVAL_BATCH):
    batch = vi_texts[start:start + EVAL_BATCH]
    preds = translate_batch(batch)
    predictions.extend(preds)
    if start % 40 == 0:
        print(f"  Processed {start}/{len(vi_texts)}...")

# Compute metrics
bleu_result = bleu_metric.compute(predictions=predictions, references=en_refs)
chrf_result = chrf_metric.compute(predictions=predictions, references=en_refs)

print(f"\n📊 Evaluation Results (VI→EN):")
print(f"   BLEU  : {bleu_result['score']:.2f}")
print(f"   chrF  : {chrf_result['score']:.2f}")

# Show some examples
print("\n📋 Sample Translations:")
for i in range(min(5, len(vi_texts))):
    print(f"\n[{i+1}] VI  : {vi_texts[i]}")
    print(f"     EN  : {en_refs[i][0]}")
    print(f"     PRED: {predictions[i]}")

## 8️⃣ Save & Export Model

In [ ]:
# ── Save LoRA adapters (lightweight — only ~100-500 MB) ──────────────────────
LORA_SAVE_PATH = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"✅ LoRA adapters saved to: {LORA_SAVE_PATH}")

# List saved files
import os
saved_files = os.listdir(LORA_SAVE_PATH)
total_size = sum(os.path.getsize(f"{LORA_SAVE_PATH}/{f}") for f in saved_files)
print(f"   Saved {len(saved_files)} files | Total size: {total_size / 1e6:.1f} MB")

In [ ]:
# ── Option A: Save merged model (full precision) ─────────────────────────────
# This merges LoRA weights into the base model for standalone inference
# WARNING: Requires ~14 GB disk space for 7B model

MERGED_SAVE_PATH = f"{OUTPUT_DIR}/merged_model"
print(f"💾 Merging & saving full model to {MERGED_SAVE_PATH}...")
print("   (This may take 5-10 minutes)")

model.save_pretrained_merged(
    MERGED_SAVE_PATH,
    tokenizer,
    save_method="merged_16bit",  # merged float16 (half the size of float32)
)
print("✅ Merged model saved!")

In [ ]:
# ── Option B: Push LoRA adapters to HuggingFace Hub ──────────────────────────
if HF_TOKEN and HF_REPO_ID != "YOUR_HF_USERNAME/edumind-vietmix-qwen2.5-7b-lora":
    from huggingface_hub import login
    login(token=HF_TOKEN)
    
    print(f"📤 Pushing LoRA adapters to Hub: {HF_REPO_ID}")
    model.push_to_hub_merged(
        HF_REPO_ID,
        tokenizer,
        save_method="lora",
        token=HF_TOKEN,
    )
    print(f"✅ Uploaded to: https://huggingface.co/{HF_REPO_ID}")
else:
    print("⚠️  Skipping Hub push (HF_TOKEN not set or HF_REPO_ID not configured)")
    print("   To push: set HF_TOKEN as Kaggle Secret and update HF_REPO_ID in cell 3")

In [ ]:
# ── Option C: Export to GGUF (for llama.cpp / Ollama local deployment) ───────
# Useful if you want to run the model locally without GPU

GGUF_SAVE_PATH = f"{OUTPUT_DIR}/gguf_model"

print("📦 Exporting to GGUF (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method="q4_k_m",  # balanced quality/size
)

gguf_files = [f for f in os.listdir(GGUF_SAVE_PATH) if f.endswith(".gguf")]
for f in gguf_files:
    size_gb = os.path.getsize(f"{GGUF_SAVE_PATH}/{f}") / 1e9
    print(f"✅ GGUF file: {f} ({size_gb:.2f} GB)")

## 9️⃣ Quick Inference Test
> Run this cell anytime to test the fine-tuned model

In [ ]:
# ── Quick inference demo ─────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)  # ensure inference mode

TEST_SENTENCES = [
    # Code-mixed sentences with technical terms (EduMIND use case)
    "Hôm nay chúng ta học về machine learning và deep learning.",
    "Model này sử dụng transformer architecture để xử lý ngôn ngữ tự nhiên.",
    "Bạn cần fine-tune model trên dataset của mình để có kết quả tốt hơn.",
    "The neural network được train trên GPU cluster với batch size là 256.",
    "Embedding vector capture được semantic meaning của từng câu.",
]

print("🎯 EduMIND Translation Demo")
print("   (Technical terms should be preserved as-is)")
print("=" * 60)

for i, sentence in enumerate(TEST_SENTENCES, 1):
    prompt = format_vi_to_en_prompt(sentence)["text"]
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    input_len = inputs.input_ids.shape[1]
    translation = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()
    
    print(f"\n[{i}] 🇻🇳 Input  : {sentence}")
    print(f"    🇬🇧 Output : {translation}")

## 🔄 API Fallback Integration
> Shows how EduMIND uses this fine-tuned model with API fallback

In [ ]:
# ── EduMIND Integration Pattern ───────────────────────────────────────────────
# This shows the .env config pattern for using the fine-tuned model

ENV_TEMPLATE = """
# ── EduMIND Translation Configuration ──────────────────────────────────────
# Option 1: Use fine-tuned LoRA model (offline, best domain accuracy)
EDUMIND_TRANSLATION_PROVIDER=huggingface
EDUMIND_TRANSLATION_MODEL=YOUR_HF_USERNAME/edumind-vietmix-qwen2.5-7b-lora
# or local path:
# EDUMIND_TRANSLATION_MODEL=/path/to/edumind-vietmix-llm/lora_adapters

# Option 2: API fallback (when no fine-tuned model available)
# EDUMIND_TRANSLATION_PROVIDER=gemini
# GOOGLE_API_KEY=your_key_here

# Option 3: OpenAI fallback
# EDUMIND_TRANSLATION_PROVIDER=openai
# OPENAI_API_KEY=your_key_here
"""

print("📋 EduMIND .env configuration template:")
print(ENV_TEMPLATE)
print("\n✅ After fine-tuning:")
print(f"   1. Update EDUMIND_TRANSLATION_MODEL in your .env")
print(f"   2. Set EDUMIND_TRANSLATION_PROVIDER=huggingface")
print(f"   3. Restart the EduMIND service")
print(f"   4. The HuggingFaceTranslationProvider will load your fine-tuned model")